In [1]:
import pandas as pd
import sqlite3
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load master dataset we created on Day 1
master = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/processed/master.csv')

In [3]:
# Connect to SQLite (creates a database file automatically)
conn = sqlite3.connect('../data/processed/olist.db')

In [4]:
# Load master dataframe into SQLite as a table called 'orders'
master.to_sql('orders', conn, if_exists='replace', index=False)

115723

In [5]:
print(f"Database ready. Total rows loaded: {len(master):,}")
print(f"Columns available: {list(master.columns)}")

Database ready. Total rows loaded: 115,723
Columns available: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'year_month', 'year', 'month', 'delivery_days', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'review_score']


In [6]:
# Query 1 — Monthly Revenue Trend
q1 = """
SELECT 
    year_month,
    ROUND(SUM(payment_value), 2)      AS revenue,
    COUNT(DISTINCT order_id)           AS total_orders,
    ROUND(AVG(payment_value), 2)       AS avg_order_value
FROM orders
GROUP BY year_month
ORDER BY year_month
"""
monthly = pd.read_sql(q1, conn)

# Remove first and last month (usually incomplete data)
monthly = monthly.iloc[1:-1]

print(monthly.to_string(index=False))

year_month    revenue  total_orders  avg_order_value
   2016-10   62185.82           265           186.19
   2016-12      19.62             1            19.62
   2017-01  178282.10           750           181.74
   2017-02  327928.86          1653           166.21
   2017-03  508767.44          2546           164.22
   2017-04  457050.31          2303           168.34
   2017-05  707042.90          3546           164.12
   2017-06  590223.90          3135           158.36
   2017-07  720446.68          3872           151.01
   2017-08  850611.08          4193           166.56
   2017-09 1003326.07          4150           199.07
   2017-10 1012420.44          4478           184.18
   2017-11 1559739.87          7289           175.43
   2017-12 1023434.55          5513           158.21
   2018-01 1383865.26          7069           164.39
   2018-02 1295740.35          6555           164.77
   2018-03 1441973.39          7003           171.99
   2018-04 1469136.33          6798           

In [7]:
# Visualise it
fig = px.bar(monthly, x='year_month', y='revenue',
    title='Monthly Revenue Trend (R$)',
    labels={'year_month': 'Month', 'revenue': 'Revenue (R$)'},
    color='revenue', color_continuous_scale='Blues')
fig.update_layout(showlegend=False, xaxis_tickangle=-45)
fig.show()

# Save for dashboard
monthly.to_csv('../data/processed/monthly_revenue.csv', index=False)
print("Saved monthly_revenue.csv")

Saved monthly_revenue.csv


In [8]:
# Query 2 — Revenue and Orders by State
q2 = """
SELECT 
    customer_state                        AS state,
    COUNT(DISTINCT order_id)              AS total_orders,
    ROUND(SUM(payment_value), 2)          AS total_revenue,
    ROUND(AVG(payment_value), 2)          AS avg_order_value,
    ROUND(AVG(delivery_days), 1)          AS avg_delivery_days
FROM orders
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10
"""
by_state = pd.read_sql(q2, conn)
print(by_state.to_string(index=False))

state  total_orders  total_revenue  avg_order_value  avg_delivery_days
   SP         40501     7456516.62           152.76                8.3
   RJ         12350     2699623.08           180.42               14.8
   MG         11354     2290457.39           169.71               11.5
   RS          5345     1118444.44           173.91               14.7
   PR          4923     1036003.69           175.77               11.5
   BA          3256      775836.28           196.41               18.7
   SC          3546      769234.50           181.47               14.5
   GO          1957      497367.84           207.67               14.9
   DF          2080      424872.44           173.63               12.5
   ES          1995      399308.36           172.26               15.2


In [9]:
# Visualise top 10 states
fig = px.bar(by_state, x='state', y='total_revenue',
    title='Top 10 States by Revenue',
    labels={'state': 'State', 'total_revenue': 'Revenue (R$)'},
    color='total_revenue', color_continuous_scale='Teal',
    text='total_orders')
fig.update_traces(texttemplate='%{text} orders', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

# Save for dashboard
by_state.to_csv('../data/processed/revenue_by_state.csv', index=False)
print("Saved revenue_by_state.csv")

Saved revenue_by_state.csv


In [10]:
# Query 3 — Category Performance
q3 = """
SELECT 
    product_category_name              AS category,
    COUNT(DISTINCT order_id)           AS total_orders,
    ROUND(SUM(payment_value), 2)       AS total_revenue,
    ROUND(AVG(review_score), 2)        AS avg_review_score,
    ROUND(AVG(delivery_days), 1)       AS avg_delivery_days
FROM orders
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name
HAVING total_orders > 100
ORDER BY total_revenue DESC
LIMIT 15
"""
by_category = pd.read_sql(q3, conn)
print(by_category.to_string(index=False))

              category  total_orders  total_revenue  avg_review_score  avg_delivery_days
       cama_mesa_banho          9272     1723932.14              3.92               12.3
          beleza_saude          8647     1625923.50              4.19               11.5
informatica_acessorios          6530     1563315.62              3.99               12.8
      moveis_decoracao          6307     1408110.04              3.96               12.4
    relogios_presentes          5495     1388699.25              4.07               12.2
         esporte_lazer          7530     1357249.46              4.16               11.7
 utilidades_domesticas          5743     1072820.85              4.12               10.5
            automotivo          3810      835782.91              4.11               11.7
    ferramentas_jardim          3448      813055.77              4.08               13.3
            cool_stuff          3559      746763.39              4.19               11.9
     moveis_escritori

In [11]:
# Scatter plot — revenue vs review score (the interesting one)
fig = px.scatter(by_category,
    x='avg_review_score', y='total_revenue',
    size='total_orders', color='avg_review_score',
    hover_name='category',
    title='Category: Revenue vs Customer Satisfaction',
    labels={'avg_review_score': 'Avg Review Score', 'total_revenue': 'Total Revenue (R$)'},
    color_continuous_scale='RdYlGn')
fig.show()

# Save for dashboard
by_category.to_csv('../data/processed/category_performance.csv', index=False)
print("Saved category_performance.csv")

Saved category_performance.csv


In [12]:
# Query 4 — Delivery Performance by State
q4 = """
SELECT
    customer_state                                        AS state,
    ROUND(AVG(delivery_days), 1)                         AS actual_delivery_days,
    COUNT(DISTINCT order_id)                              AS total_orders,
    ROUND(AVG(review_score), 2)                          AS avg_review_score
FROM orders
WHERE delivery_days IS NOT NULL
  AND delivery_days > 0
GROUP BY customer_state
HAVING total_orders > 50
ORDER BY actual_delivery_days DESC
"""
delivery = pd.read_sql(q4, conn)
print(delivery.to_string(index=False))

state  actual_delivery_days  total_orders  avg_review_score
   AP                  27.7            67              4.28
   AM                  26.1           145              4.10
   AL                  24.1           397              3.81
   PA                  23.3           946              3.84
   MA                  21.2           717              3.74
   SE                  20.9           335              3.90
   CE                  20.3          1279              3.88
   AC                  20.2            80              4.13
   PB                  20.1           517              4.04
   RO                  19.2           243              4.09
   RN                  19.2           474              4.09
   PI                  18.9           476              3.92
   BA                  18.8          3255              3.87
   PE                  17.8          1593              4.03
   MT                  17.5           886              4.00
   TO                  16.7           27

In [13]:
# Bar chart — delivery days by state
fig = px.bar(delivery.head(15), x='state', y='actual_delivery_days',
    title='Average Delivery Days by State (Top 15 Slowest)',
    labels={'state': 'State', 'actual_delivery_days': 'Avg Delivery Days'},
    color='actual_delivery_days', color_continuous_scale='Reds')
fig.update_layout(showlegend=False)
fig.show()

# Save for dashboard
delivery.to_csv('../data/processed/delivery_performance.csv', index=False)
print("Saved delivery_performance.csv")

Saved delivery_performance.csv


In [14]:
# Query 5 — Payment Method Analysis
q5 = """
SELECT
    payment_type,
    COUNT(DISTINCT order_id)          AS total_orders,
    ROUND(SUM(payment_value), 2)      AS total_revenue,
    ROUND(AVG(payment_value), 2)      AS avg_order_value,
    ROUND(AVG(payment_installments),1)AS avg_instalments
FROM orders
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_orders DESC
"""
payments_analysis = pd.read_sql(q5, conn)
print(payments_analysis.to_string(index=False))

payment_type  total_orders  total_revenue  avg_order_value  avg_instalments
 credit_card         74304    15268681.44           178.85              3.6
      boleto         19191     3966153.08           176.23              1.0
     voucher          3679      399426.07            64.62              1.0
  debit_card          1485      247684.48           149.03              1.0


In [15]:
# Pie chart — payment type split
fig = px.pie(payments_analysis, names='payment_type', values='total_orders',
    title='Orders by Payment Method',
    color_discrete_sequence=px.colors.sequential.Blues_r)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

# Save for dashboard
payments_analysis.to_csv('../data/processed/payment_analysis.csv', index=False)
print("Saved payment_analysis.csv")

Saved payment_analysis.csv
